<a href="https://colab.research.google.com/github/xesmaze/CPSC499-MDV-SR25/blob/main/Writing_Regression_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Writing Statistical Models

The same experimental design can be written in several equivalent ways.  
We will practice translating among:

1. **Experimental description**
2. **Scalar model notation**
3. **Matrix notation**
4. **R / lme4-style formula notation**

---

## Model Writing Reference Table

| Experimental Design | Scalar Model | Matrix Form | R / lme4 Formula |
|---|---|---|---|
| **Completely Randomized Design (CRD)** | $Y_{ij}=\mu+T_i+\epsilon_{ij}$ | $\mathbf{y}=\mathbf{1}\mu+\mathbf{X}_T\mathbf{T}+\boldsymbol{\epsilon}$ | `Y ~ T` |
| **Randomized Complete Block Design (RCBD)** | $Y_{ij}=\mu+T_i+B_j+\epsilon_{ij}$ | $\mathbf{y}=\mathbf{1}\mu+\mathbf{X}_T\mathbf{T}+\mathbf{Z}_B\mathbf{B}+\boldsymbol{\epsilon}$ | `Y ~ T + (1 \| B)` |
| **Two-factor factorial** | $Y_{ijk}=\mu+A_i+B_j+(AB)_{ij}+\epsilon_{ijk}$ | $\mathbf{y}=\mathbf{1}\mu+\mathbf{X}_A\mathbf{A}+\mathbf{X}_B\mathbf{B}+\mathbf{X}_{AB}\mathbf{AB}+\boldsymbol{\epsilon}$ | `Y ~ A * B` |
| **Factorial + random block** | $Y_{ijkl}=\mu+A_i+B_j+(AB)_{ij}+R_k+\epsilon_{ijkl}$ | $\mathbf{y}=\mathbf{1}\mu+\mathbf{X}_A\mathbf{A}+\mathbf{X}_B\mathbf{B}+\mathbf{X}_{AB}\mathbf{AB}+\mathbf{Z}_R\mathbf{R}+\boldsymbol{\epsilon}$ | `Y ~ A * B + (1 \| R)` |
| **Blocks nested within environment** | $Y_{ijk}=\mu+G_i+E_j+(GE)_{ij}+B_{k(j)}+\epsilon_{ijk}$ | $\mathbf{y}=\mathbf{1}\mu+\mathbf{X}_G\mathbf{G}+\mathbf{X}_E\mathbf{E}+\mathbf{X}_{GE}\mathbf{GE}+\mathbf{Z}_{B:E}\mathbf{B}+\boldsymbol{\epsilon}$ | `Y ~ G * E + (1 \| E:B)` |
| **Hierarchical environment/block model** | $Y_{ijk}=\mu+G_i+E_j+(GE)_{ij}+B_{k(j)}+\epsilon_{ijk}$ | $\mathbf{y}=\mathbf{X}\mathbf{b}+\mathbf{Z}\mathbf{u}+\boldsymbol{\epsilon}$ | `Y ~ G * E + (1 \| E) + (1 \| E:B)` |
| **Random intercept and random slope** | $Y_{ij}=\mu+X_{ij}+u_{0j}+u_{1j}X_{ij}+\epsilon_{ij}$ | $\mathbf{y}=\mathbf{X}\mathbf{b}+\mathbf{Z}\mathbf{u}+\boldsymbol{\epsilon}$ | `Y ~ X + (X \| B)` |
| **Random slope without random intercept** | $Y_{ij}=\mu+X_{ij}+u_{1j}X_{ij}+\epsilon_{ij}$ | $\mathbf{y}=\mathbf{X}\mathbf{b}+\mathbf{Z}_X\mathbf{u}_X+\boldsymbol{\epsilon}$ | `Y ~ X + (0 + X \| B)` |

---

## Reading the Scalar Model

Consider the randomized complete block model:

$$
Y_{ij}=\mu+T_i+B_j+\epsilon_{ij}
$$

where:

- $Y_{ij}$ = observed response for treatment $i$ in block $j$
- $\mu$ = overall mean
- $T_i$ = effect of treatment $i$
- $B_j$ = effect of block $j$
- $\epsilon_{ij}$ = residual variation not explained by the model

The scalar equation describes the model **one observation at a time**.

---

## Reading the Matrix Model

The same RCBD model can be written as:

$$
\mathbf{y}
=
\mathbf{1}\mu
+
\mathbf{X}_T\mathbf{T}
+
\mathbf{Z}_B\mathbf{B}
+
\boldsymbol{\epsilon}
$$

where:

- $\mathbf{y}$ = vector containing all observations
- $\mathbf{1}\mu$ = overall mean
- $\mathbf{X}_T$ = design matrix connecting observations to treatments
- $\mathbf{T}$ = vector of treatment effects
- $\mathbf{Z}_B$ = incidence matrix connecting observations to blocks
- $\mathbf{B}$ = vector of block effects
- $\boldsymbol{\epsilon}$ = vector of residuals

The matrix equation describes **all observations simultaneously**.

---

## Fixed and Random Effects

A useful way to think about the matrix model is:

$$
\mathbf{y}
=
\text{fixed effects}
+
\text{random effects}
+
\text{residual}
$$

or, in compact mixed-model notation,

$$
\mathbf{y}
=
\mathbf{X}\mathbf{b}
+
\mathbf{Z}\mathbf{u}
+
\boldsymbol{\epsilon}
$$

where:

- $\mathbf{X}\mathbf{b}$ contains the **fixed effects**
- $\mathbf{Z}\mathbf{u}$ contains the **random effects**
- $\boldsymbol{\epsilon}$ contains unexplained residual variation

For example,

$$
Y_{ij}=\mu+T_i+B_j+\epsilon_{ij}
$$

with treatment fixed and block random becomes

$$
\mathbf{y}
=
\mathbf{1}\mu
+
\mathbf{X}_T\mathbf{T}
+
\mathbf{Z}_B\mathbf{B}
+
\boldsymbol{\epsilon}
$$

and in R:

`Y ~ T + (1 | B)`

---

## Interaction

An interaction asks whether the effect of one factor depends on the level of another factor.

For example:

$$
Y_{ijk}
=
\mu
+
G_i
+
E_j
+
(GE)_{ij}
+
\epsilon_{ijk}
$$

The interaction term is:

$$
(GE)_{ij}
$$

In matrix notation:

$$
\mathbf{X}_{GE}\mathbf{GE}
$$

In formula notation:

`G * E`

In R, the expression

`G * E`

expands to:

`G + E + G:E`

Thus,

$$
G\times E
$$

represents the biological/statistical interaction between genotype and environment.

---

## Nested Effects

Nesting occurs when the levels of one factor exist only within the levels of another factor.

For example, suppose every environment contains its own four blocks.

Then Block 1 at Environment 1 is **not the same experimental block** as Block 1 at Environment 2.

The scalar model can be written:

$$
Y_{ijk}
=
\mu
+
G_i
+
E_j
+
(GE)_{ij}
+
B_{k(j)}
+
\epsilon_{ijk}
$$

where

$$
B_{k(j)}
$$

means:

> block $k$ nested within environment $j$

We can represent the same structure as:

$$
B:E
$$

or conceptually as:

$$
B|E
$$

The matrix contribution is:

$$
\mathbf{Z}_{B:E}\mathbf{B}
$$

and an lme4 random-intercept specification is:

`(1 | E:B)`

---

## `:` versus `/` in R

The colon constructs the combination of two factors:

`E:B`

This identifies each unique Environment × Block combination.

For example:

- E1:B1
- E1:B2
- E2:B1
- E2:B2

Therefore:

`(1 | E:B)`

fits a random intercept for each Environment × Block combination.

The slash notation:

`E/B`

expands to:

`E + E:B`

Therefore:

`(1 | E/B)`

expands to:

`(1 | E) + (1 | E:B)`

This model includes:

1. variation among environments
2. variation among blocks within environments

---

## Random Intercepts

The notation

`(1 | B)`

means that each level of `B` receives its own random intercept.

For example:

$$
Y_{ij}
=
\mu
+
T_i
+
u_j
+
\epsilon_{ij}
$$

where:

$$
u_j \sim N(0,\sigma_B^2)
$$

In R:

`Y ~ T + (1 | B)`

---

## Random Slopes

Suppose the response to $X$ is allowed to vary among groups $B$.

The model is:

$$
Y_{ij}
=
\mu
+
\beta X_{ij}
+
u_{0j}
+
u_{1j}X_{ij}
+
\epsilon_{ij}
$$

In lme4:

`Y ~ X + (X | B)`

This estimates:

- a random intercept for each level of `B`
- a random slope for `X` within each level of `B`

---

## Random Slope Without a Random Intercept

If we want the effect of $X$ to vary among groups but do **not** want a random intercept:

`Y ~ X + (0 + X | B)`

or equivalently:

`Y ~ X + (-1 + X | B)`

The `0 +` or `-1 +` removes the random intercept from that random-effect term.

---

## Model-Writing Workflow

For every experiment, work through the model in this order:

### 1. Identify the response

What was measured?

$$
Y
$$

### 2. Identify the experimental factors

What variables are being compared?

For example:

$$
G,\;E,\;N,\;W
$$

### 3. Identify interactions

Does the effect of one factor depend on another?

For example:

$$
G\times E
$$

### 4. Identify the experimental design structure

Are observations grouped into:

- blocks?
- locations?
- years?
- plots?
- repeated measurements?

### 5. Identify nesting

Does one factor exist only within another?

For example:

$$
B_{k(j)}
$$

means blocks are nested within environments.

### 6. Decide which effects are fixed and random

Then translate the model into:

$$
\mathbf{y}
=
\mathbf{X}\mathbf{b}
+
\mathbf{Z}\mathbf{u}
+
\boldsymbol{\epsilon}
$$

and finally into R / lme4 notation.

---

## Translation Pattern

A useful way to think about model writing is:

$$
\boxed{
\text{Experimental Design}
\rightarrow
\text{Scalar Model}
\rightarrow
\text{Matrix Model}
\rightarrow
\text{R Formula}
}
$$

For example:

**Experiment:** Genotypes evaluated across environments with blocks nested within environments

$$
\Downarrow
$$

**Scalar**

$$
Y_{ijk}
=
\mu+G_i+E_j+(GE)_{ij}+B_{k(j)}+\epsilon_{ijk}
$$

$$
\Downarrow
$$

**Matrix**

$$
\mathbf{y}
=
\mathbf{1}\mu
+
\mathbf{X}_G\mathbf{G}
+
\mathbf{X}_E\mathbf{E}
+
\mathbf{X}_{GE}\mathbf{GE}
+
\mathbf{Z}_{B:E}\mathbf{B}
+
\boldsymbol{\epsilon}
$$

$$
\Downarrow
$$

**R / lme4**

`Y ~ G * E + (1 | E:B)`